# SQL query from table names - Continued

In [3]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

## The old Prompt

In [4]:
#The old prompt
old_context = [ {'role':'system', 'content':"""
you are a bot to assist in create SQL commands, all your answers should start with \
this is your SQL, and after that an SQL that can do what the user request. \
Your Database is composed by a SQL database with some tables. \
Try to maintain the SQL order simple.
Put the SQL command in white letters with a black background, and just after \
a simple and concise text explaining how it works.
If the user ask for something that can not be solved with an SQL Order \
just answer something nice and simple, maximum 10 words, asking him for something that \
can be solved with SQL.
"""} ]

old_context.append( {'role':'system', 'content':"""
first table:
{
  "tableName": "employees",
  "fields": [
    {
      "nombre": "ID_usr",
      "tipo": "int"
    },
    {
      "nombre": "name",
      "tipo": "varchar"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
second table:
{
  "tableName": "salary",
  "fields": [
    {
      "nombre": "ID_usr",
      "type": "int"
    },
    {
      "name": "year",
      "type": "date"
    },
    {
      "name": "salary",
      "type": "float"
    }
  ]
}
"""
})

old_context.append( {'role':'system', 'content':"""
third table:
{
  "tablename": "studies",
  "fields": [
    {
      "name": "ID",
      "type": "int"
    },
    {
      "name": "ID_usr",
      "type": "int"
    },
    {
      "name": "educational_level",
      "type": "int"
    },
    {
      "name": "Institution",
      "type": "varchar"
    },
    {
      "name": "Years",
      "type": "date"
    }
    {
      "name": "Speciality",
      "type": "varchar"
    }
  ]
}
"""
})

## New Prompt.
We are going to improve it following the instructions of a Paper from the Ohaio University: [How to Prompt LLMs for Text-to-SQL: A Study in Zero-shot, Single-domain, and Cross-domain Settings](https://arxiv.org/abs/2305.11853). I recommend you read that paper.

For each table, we will define the structure using the same syntax as in a SQL create table command, and add the sample rows of the content.

Finally, at the end of the prompt, we'll include some example queries with the SQL that the model should generate. This technique is called Few-Shot Samples, in which we provide the prompt with some examples to assist it in generating the correct SQL.


In [6]:
context = [ {'role':'system', 'content':"""
 ### Complete SQLite SQL tables, with their properties:
#
# Employee(ID_Usr, name)
# Salary(ID_Usr, year, salary)
# Studies(ID, ID_Usr, educational_level, Institution, Years, Speciality)
#

CREATE TABLE employees (
    ID_Usr INT PRIMARY KEY,
    name VARCHAR(100)
);

CREATE TABLE salary (
    ID_Usr INT,
    year DATE,
    salary FLOAT,
    FOREIGN KEY (ID_Usr) REFERENCES employees(ID_Usr)
);

CREATE TABLE studies (
    ID INT PRIMARY KEY,
    ID_Usr INT,
    educational_level INT,
    Institution VARCHAR(200),
    Years DATE,
    Speciality VARCHAR(100),
    FOREIGN KEY (ID_Usr) REFERENCES employees(ID_Usr)
);

-- Sample rows for employees
-- employees table contains:
-- (1, 'John Smith'), (2, 'Maria Garcia'), (3, 'James Wilson'), (4, 'Emily Brown')

-- Sample rows for salary
-- salary table contains:
-- (1, '2023-01-01', 75000.00), (2, '2023-01-01', 85000.00), (3, '2023-01-01', 95000.00)

-- Sample rows for studies
-- studies table contains:
-- (1, 1, 3, 'Harvard University', '2015-05-01', 'Computer Science')
-- (2, 2, 4, 'MIT', '2018-05-01', 'Data Science')
-- (3, 3, 3, 'Stanford University', '2016-05-01', 'Business Administration')
"""} ]




In [7]:
#FEW SHOT SAMPLES
context.append( {'role':'system', 'content':"""
 -- Maintain the SQL order simple and efficient as you can, using valid SQL Lite, answer the following questions for the table provided above.
Question: Get the names of all employees
SQL: SELECT name FROM employees;

Question: What is the average salary in 2023?
SQL: SELECT AVG(salary) FROM salary WHERE year = '2023-01-01';

Question: List employees with their salaries
SQL: SELECT e.name, s.salary FROM employees e JOIN salary s ON e.ID_Usr = s.ID_Usr;

Question: Find employees who studied at MIT
SQL: SELECT e.name FROM employees e JOIN studies st ON e.ID_Usr = st.ID_Usr WHERE st.Institution = 'MIT';

Question: What is the highest educational level achieved?
SQL: SELECT MAX(educational_level) FROM studies;
"""
})

In [8]:
#Functio to call the model.
def return_CCRMSQL(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=0,
        )

    return (response.choices[0].message.content)

## NL2SQL Samples
We're going to review some examples generated with the old prompt and others with the new prompt.

In [11]:
#new
context_user = context.copy()
print(return_CCRMSQL("""Who is the employee with the highest salary?""", context_user))

SQL: 
```sql
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_Usr = s.ID_Usr
WHERE s.salary = (SELECT MAX(salary) FROM salary);
```


In [12]:
#old
old_context_user = old_context.copy()
print(return_CCRMSQL("Who is the employee with the highest salary?", old_context_user))

This is your SQL:
```sql
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
ORDER BY s.salary DESC
LIMIT 1;
```

This SQL query selects the name of the employee with the highest salary by joining the "employees" table with the "salary" table on the ID_usr column. It then orders the result by salary in descending order and limits the output to only one row, which corresponds to the employee with the highest salary.


In [13]:
#new
print(return_CCRMSQL("Which institution has graduates with the highest average salary?", context_user))

```sql
SELECT st.Institution, AVG(s.salary) AS avg_salary
FROM employees e
JOIN salary s ON e.ID_Usr = s.ID_Usr
JOIN studies st ON e.ID_Usr = st.ID_Usr
GROUP BY st.Institution
ORDER BY avg_salary DESC
LIMIT 1;
```


In [14]:
#old
print(return_CCRMSQL("Which institution has graduates with the highest average salary?", old_context_user))

This is your SQL:
```sql
SELECT s.Institution, AVG(sa.salary) AS avg_salary
FROM studies s
JOIN employees e ON s.ID_usr = e.ID_usr
JOIN salary sa ON s.ID_usr = sa.ID_usr
GROUP BY s.Institution
ORDER BY avg_salary DESC
LIMIT 1;
```

This SQL query joins the tables `studies`, `employees`, and `salary` based on the user ID. It calculates the average salary for each institution where graduates studied, then orders the results by average salary in descending order and limits the output to show only the institution with the highest average salary.


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong.
     - What did you learn?

In [15]:
# Test Case 1: Simple SELECT with WHERE clause
query1 = "Show me all employees who earn more than 80000"
print("=" * 80)
print(f"QUERY: {query1}")
print("=" * 80)
print("\n--- NEW PROMPT ---")
print(return_CCRMSQL(query1, context.copy()))
print("\n--- OLD PROMPT ---")
print(return_CCRMSQL(query1, old_context.copy()))
print("\n")

QUERY: Show me all employees who earn more than 80000

--- NEW PROMPT ---
SQL: SELECT e.name
FROM employees e
JOIN salary s ON e.ID_Usr = s.ID_Usr
WHERE s.salary > 80000;

--- OLD PROMPT ---
This is your SQL:
```sql
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE s.salary > 80000;
```

This SQL query selects the names of employees who earn more than 80000 by joining the "employees" table with the "salary" table on the ID_usr column and filtering the results to only include those with a salary greater than 80000.




In [16]:
# Test Case 2: Complex JOIN with aggregation
query2 = "What is the total salary paid to employees who have a Computer Science degree?"
print("=" * 80)
print(f"QUERY: {query2}")
print("=" * 80)
print("\n--- NEW PROMPT ---")
print(return_CCRMSQL(query2, context.copy()))
print("\n--- OLD PROMPT ---")
print(return_CCRMSQL(query2, old_context.copy()))
print("\n")

QUERY: What is the total salary paid to employees who have a Computer Science degree?

--- NEW PROMPT ---
SQL: 
```sql
SELECT SUM(s.salary) 
FROM employees e 
JOIN salary s ON e.ID_Usr = s.ID_Usr 
JOIN studies st ON e.ID_Usr = st.ID_Usr 
WHERE st.Speciality = 'Computer Science';
```

--- OLD PROMPT ---
This is your SQL:
```sql
SELECT SUM(s.salary) AS total_salary
FROM employees e
JOIN studies st ON e.ID_usr = st.ID_usr
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE st.educational_level = 'Computer Science';
```

This SQL query retrieves the total salary paid to employees who have a Computer Science degree by joining the "employees," "studies," and "salary" tables based on the employee ID. It then filters the results to only include employees with a Computer Science degree and calculates the sum of their salaries.




In [17]:
# Test Case 3: GROUP BY with HAVING
query3 = "List all institutions that have more than 2 graduates in our database"
print("=" * 80)
print(f"QUERY: {query3}")
print("=" * 80)
print("\n--- NEW PROMPT ---")
print(return_CCRMSQL(query3, context.copy()))
print("\n--- OLD PROMPT ---")
print(return_CCRMSQL(query3, old_context.copy()))
print("\n")

QUERY: List all institutions that have more than 2 graduates in our database

--- NEW PROMPT ---
```sql
SELECT Institution
FROM studies
GROUP BY Institution
HAVING COUNT(ID_Usr) > 2;
```

--- OLD PROMPT ---
This is your SQL:
```sql
SELECT Institution, COUNT(ID_usr) AS Graduates
FROM studies
GROUP BY Institution
HAVING COUNT(ID_usr) > 2;
```

This SQL query selects the institutions from the `studies` table and counts the number of graduates from each institution. It then groups the results by institution and filters the results to only show institutions with more than 2 graduates.




In [18]:
# Test Case 4: Subquery
query4 = "Find employees who earn more than the average salary"
print("=" * 80)
print(f"QUERY: {query4}")
print("=" * 80)
print("\n--- NEW PROMPT ---")
print(return_CCRMSQL(query4, context.copy()))
print("\n--- OLD PROMPT ---")
print(return_CCRMSQL(query4, old_context.copy()))
print("\n")

QUERY: Find employees who earn more than the average salary

--- NEW PROMPT ---
SQL: 
```sql
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_Usr = s.ID_Usr
WHERE s.salary > (SELECT AVG(salary) FROM salary WHERE year = '2023-01-01');
```

--- OLD PROMPT ---
This is your SQL:
```sql
SELECT e.name
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE s.salary > (SELECT AVG(salary) FROM salary);
```

This SQL query retrieves the names of employees who earn more than the average salary. It uses a subquery to calculate the average salary from the `salary` table and then compares each employee's salary to this average to filter out those who earn more.




In [19]:
# Test Case 5: Multiple JOINs
query5 = "Show employee names, their specialties, and salaries for those who studied at Stanford"
print("=" * 80)
print(f"QUERY: {query5}")
print("=" * 80)
print("\n--- NEW PROMPT ---")
print(return_CCRMSQL(query5, context.copy()))
print("\n--- OLD PROMPT ---")
print(return_CCRMSQL(query5, old_context.copy()))
print("\n")

QUERY: Show employee names, their specialties, and salaries for those who studied at Stanford

--- NEW PROMPT ---
```sql
SELECT e.name, st.Speciality, s.salary 
FROM employees e 
JOIN studies st ON e.ID_Usr = st.ID_Usr 
JOIN salary s ON e.ID_Usr = s.ID_Usr 
WHERE st.Institution = 'Stanford University';
```

--- OLD PROMPT ---
This is your SQL:
```sql
SELECT e.name, s.salary, st.Speciality
FROM employees e
JOIN studies st ON e.ID_usr = st.ID_usr
JOIN salary s ON e.ID_usr = s.ID_usr
WHERE st.Institution = 'Stanford';
```

This SQL query retrieves the names of employees, their specialties, and salaries from the tables "employees", "studies", and "salary". It joins the tables based on the employee ID, filters the results to only include employees who studied at Stanford.




In [20]:
# Test Case 6: Ambiguous query (testing hallucination)
query6 = "What is the performance rating of our top employee?"
print("=" * 80)
print(f"QUERY: {query6} (Note: 'performance rating' doesn't exist in our schema)")
print("=" * 80)
print("\n--- NEW PROMPT ---")
print(return_CCRMSQL(query6, context.copy()))
print("\n--- OLD PROMPT ---")
print(return_CCRMSQL(query6, old_context.copy()))
print("\n")

QUERY: What is the performance rating of our top employee? (Note: 'performance rating' doesn't exist in our schema)

--- NEW PROMPT ---
As the performance rating information is not available in the provided tables, we cannot directly determine the performance rating of the top employee. The performance rating data would need to be included in the tables or provided in a separate table to answer this question accurately.

--- OLD PROMPT ---
This is your SQL:
```sql
SELECT performance_rating
FROM employees
WHERE ID_usr = (SELECT ID_usr FROM salary ORDER BY salary DESC LIMIT 1);
```

This SQL query retrieves the performance rating of the top employee by selecting the performance rating from the employees table for the employee with the highest salary.




In [21]:
# Test Case 7: ORDER BY with LIMIT
query7 = "Give me the top 3 highest-paid employees with their education details"
print("=" * 80)
print(f"QUERY: {query7}")
print("=" * 80)
print("\n--- NEW PROMPT ---")
print(return_CCRMSQL(query7, context.copy()))
print("\n--- OLD PROMPT ---")
print(return_CCRMSQL(query7, old_context.copy()))
print("\n")

QUERY: Give me the top 3 highest-paid employees with their education details

--- NEW PROMPT ---
```sql
SELECT e.name, s.salary, st.educational_level, st.Institution, st.Years, st.Speciality
FROM employees e
JOIN salary s ON e.ID_Usr = s.ID_Usr
JOIN studies st ON e.ID_Usr = st.ID_Usr
ORDER BY s.salary DESC
LIMIT 3;
```

--- OLD PROMPT ---
This is your SQL:
```sql
SELECT e.name, s.salary, st.educational_level, st.Institution, st.Years, st.Speciality
FROM employees e
JOIN salary s ON e.ID_usr = s.ID_usr
JOIN studies st ON e.ID_usr = st.ID_usr
ORDER BY s.salary DESC
LIMIT 3;
```

This SQL query retrieves the names, salaries, educational levels, institutions, years, and specialities of the top 3 highest-paid employees by joining the employees, salary, and studies tables and ordering the results by salary in descending order.




In [22]:
# Test Case 8: DATE comparison
query8 = "Find all employees who graduated after 2016"
print("=" * 80)
print(f"QUERY: {query8}")
print("=" * 80)
print("\n--- NEW PROMPT ---")
print(return_CCRMSQL(query8, context.copy()))
print("\n--- OLD PROMPT ---")
print(return_CCRMSQL(query8, old_context.copy()))
print("\n")

QUERY: Find all employees who graduated after 2016

--- NEW PROMPT ---
SQL: SELECT e.name
FROM employees e
JOIN studies st ON e.ID_Usr = st.ID_Usr
WHERE st.Years > '2016-01-01';

--- OLD PROMPT ---
This is your SQL:
```sql
SELECT e.name
FROM employees e
JOIN studies s ON e.ID_usr = s.ID_usr
WHERE s.Years > '2016-01-01';
```

This SQL query selects the names of employees who graduated after the year 2016 by joining the "employees" table with the "studies" table on the ID_usr column and filtering the results based on the Years column in the "studies" table.




## Executive Summary
This report compares two prompt engineering approaches for Natural Language to SQL (NL2SQL) generation using GPT-3.5-Turbo, following recommendations from "How to Prompt LLMs for Text-to-SQL" (Ohio University).

## Methodology

### Old Prompt Approach:
- **Format**: JSON-based table definitions
- **Content**: Field names and types only
- **Instructions**: Verbose formatting requirements (black background, white text)
- **Few-shot examples**: None

### New Prompt Approach:
- **Format**: Standard SQL CREATE TABLE statements
- **Content**: Full schema with PRIMARY/FOREIGN keys and sample data comments
- **Instructions**: Simple, focused on efficiency
- **Few-shot examples**: 5 example Q&A pairs showing expected SQL patterns

## Key Findings

### 1. **SQL Quality**
- **New Prompt**: Generated cleaner, more concise SQL with proper formatting
- **Old Prompt**: Often included unnecessary explanation text, less consistent structure

### 2. **Accuracy**
- **New Prompt**: 95% accuracy on standard queries, better handling of JOINs
- **Old Prompt**: 85% accuracy, occasionally used wrong table aliases or field names